<a href="https://colab.research.google.com/github/syrashid/rl-lunar-lander/blob/main/sandbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Update apt, install dependencies, install**

In [1]:
!apt-get -y update
!apt-get -y install swig cmake ffmpeg
!pip -q install "gymnasium[box2d]" imageio imageio-ffmpeg

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,885 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,288 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,668 kB]
Get:14

**Set up Lunar Lander env + video recording**

In [3]:
import os
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

VIDEO_DIR = "/content/lunar_lander/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

# LunarLander is usually v3, but this fallback keeps you unblocked.
def make_lander_env(render_mode="rgb_array"):
    for env_id in ["LunarLander-v2", "LunarLander-v3"]:
        try:
            return gym.make(env_id, render_mode=render_mode), env_id
        except Exception:
            pass
    raise RuntimeError("Could not create LunarLander env (tried v2/v3/v4).")

env, ENV_ID = make_lander_env()

env = RecordVideo(
    env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda ep_idx: True,  # record every episode
    name_prefix=f"{ENV_ID}_random"
)

print("Using env:", ENV_ID)
print("Recording to:", VIDEO_DIR)

Using env: LunarLander-v3
Recording to: /content/lunar_lander/videos


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment LunarLander-v2 is out of date. You should consider upgrading to version `v3`.
  logger.deprecation(
/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/lunar_lander/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


**Run random episodes**

In [10]:
def run_episodes(env, n_episodes=3, seed=42, max_steps=2000):
    returns = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed + ep)
        done = False
        truncated = False
        ep_return = 0.0
        steps = 0

        while not (done or truncated) and steps < max_steps:
            action = env.action_space.sample()   # random for now
            obs, reward, done, truncated, info = env.step(action)
            ep_return += float(reward)
            steps += 1

        returns.append(ep_return)
        print(f"Episode {ep}: return={ep_return:.1f}, steps={steps}, done={done}, truncated={truncated}")

    return returns

returns = run_episodes(env, n_episodes=3)
env.close()

print("Returns:", returns)

Episode 0: return=-162.2, steps=110, done=True, truncated=False
Episode 1: return=-130.6, steps=75, done=True, truncated=False
Episode 2: return=-209.8, steps=121, done=True, truncated=False
Returns: [-162.21981126863136, -130.64176301989374, -209.75333237225252]


**List recordings**

In [11]:
import glob
videos = sorted(glob.glob(f"{VIDEO_DIR}/*.mp4"))
videos

['/content/lunar_lander/videos/LunarLander-v3_random-episode-0.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-1.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-2.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-3.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-4.mp4',
 '/content/lunar_lander/videos/LunarLander-v3_random-episode-5.mp4']

**View video display**

In [16]:
from IPython.display import Video, display

latest = videos[-2]
display(Video(latest, embed=True))
print(f"Displaying: {latest}")

Displaying: /content/lunar_lander/videos/LunarLander-v3_random-episode-5.mp4
